# HEMR Dataset Preprocessing Pipeline
## Last.fm + Echo Nest (Million Song Dataset)

**Purpose:** Transform raw Last.fm and Echo Nest datasets into the exact format required by the HEMR hypergraph-based music recommendation notebooks.

### Target Output Format (derived from notebook analysis)

| File | Columns | Notes |
|---|---|---|
| `songs_final.csv` | `song_id`, `artist_name`, `release`, `title`, `tags`, `arousal`, `valence`, `lyrics_text` | One row per unique song |
| `user_data_final.csv` | `user`, `song_id`, `play_count` | One row per (user, song) pair |
| `user_data_partitioned_val.csv` | above + `triple_id`, `set` | With train/test split (80/20 per user) |

### Raw Input Files (in Google Drive folder `HEMR_raw`)
- `train_triplets.txt.zip` — Echo Nest taste profile: `user_id \t song_id \t play_count`
- `track_metadata.db` — SQLite DB: MSD track metadata (song_id, title, release, artist_name, …)
- `lastfm_train.zip` — Last.fm listening log or tag data
- `lastfm_test.zip` — Last.fm listening log or tag data (test split)

### Resource Optimisation Strategy
- Chunked reading with `pd.read_csv(chunksize=…)` for large text files
- Downcast dtypes immediately (`int32`, `float32`, `category`)
- Process SQLite in batches via `LIMIT/OFFSET`
- Garbage-collect intermediate objects after each major step
- Never load the full 1 GB train file into RAM at once

> **NOTE on `arousal`, `valence`, `lyrics_text`:** These columns are used by the downstream notebooks
> but come from *separate* external datasets (e.g. MER / Emotify for mood, musiXmatch for lyrics).
> This notebook fills them with sensible defaults (`0.5` / `""`) when those sources are unavailable,
> and provides optional cells to merge them in if you have the files.

## 0. Setup & Configuration

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Path Configuration ─────────────────────────────────────────────────────
# Change RAW_DIR to match your actual Google Drive path
RAW_DIR    = "drive/MyDrive/HEMR_raw"
OUTPUT_DIR = "drive/MyDrive/HEMR_preprocessed"

# Raw file paths
TRIPLETS_ZIP      = f"{RAW_DIR}/train_triplets.txt.zip"
TRACK_META_DB     = f"{RAW_DIR}/track_metadata.db"
LASTFM_TRAIN_ZIP  = f"{RAW_DIR}/lastfm_train.zip"
LASTFM_TEST_ZIP   = f"{RAW_DIR}/lastfm_test.zip"

# Output file paths (matching names expected by the HEMR notebooks)
OUT_SONGS         = f"{OUTPUT_DIR}/songs_final.csv"
OUT_USER_DATA     = f"{OUTPUT_DIR}/user_data_final.csv"
OUT_USER_PART     = f"{OUTPUT_DIR}/user_data_partitioned_val.csv"

# ── Preprocessing Parameters ───────────────────────────────────────────────
CHUNK_SIZE        = 100_000   # rows per chunk for large file reads
MIN_PLAY_COUNT    = 1         # minimum play count to keep an interaction
MIN_SONGS_PER_USER = 2        # minimum songs a user must have listened to
TRAIN_RATIO       = 0.80      # fraction of each user's history used for training
MIN_TRAIN_ITEMS   = 5         # users with fewer interactions go entirely to train
RANDOM_SEED       = 42

import os, warnings
warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: drive/MyDrive/HEMR_preprocessed


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import gc
import sqlite3
import zipfile
import random
import numpy  as np
import pandas as pd
from tqdm.notebook import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Helper: show current memory footprint of a DataFrame
def df_mem(df, name="df"):
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"  [{name}]  shape={df.shape}  mem={mb:.1f} MB")

print("Imports OK.")

Imports OK.


## 1. Data Exploration — Understand Raw File Structures

In [ ]:
# ── Explore: Echo Nest train_triplets.txt ──────────────────────────────────
# Expected format: user_id <TAB> song_id <TAB> play_count  (no header)

print("=== Echo Nest Train Triplets ===")

with zipfile.ZipFile(TRIPLETS_ZIP) as zf:
    inner_name = zf.namelist()[0]          # e.g. 'train_triplets.txt'
    print(f"  File inside zip: {inner_name}")
    with zf.open(inner_name) as f:
        sample_trips = pd.read_csv(
            f, sep='\t', header=None,
            names=['user', 'song_id', 'play_count'],
            nrows=5
        )

print(sample_trips)
print("\nDtypes:", sample_trips.dtypes.to_dict())

=== Echo Nest Train Triplets ===
  File inside zip: train_triplets.txt
                                       user             song_id  play_count
0  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAKIMP12A8C130995           1
1  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOAPDEY12A81C210A9           1
2  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBBMDR12A8C13253B           2
3  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFNSP12AF72A0E22           1
4  b80344d063b5ccb3212f76538f3d9e43d87dca9e  SOBFOVM12A58A7D494           1

Dtypes: {'user': dtype('O'), 'song_id': dtype('O'), 'play_count': dtype('int64')}


In [ ]:
# ── Explore: track_metadata.db ────────────────────────────────────────────
print("=== Track Metadata DB ===")

conn = sqlite3.connect(TRACK_META_DB)

# List tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables:", tables['name'].tolist())

# Inspect first table
table_name = tables['name'].iloc[0]
schema = pd.read_sql(f"PRAGMA table_info({table_name})", conn)
print(f"\nSchema of '{table_name}':")
print(schema[['name','type']].to_string(index=False))

# Sample rows
sample_meta = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5", conn)
print("\nSample rows:")
print(sample_meta)

# Count total rows
total = pd.read_sql(f"SELECT COUNT(*) as n FROM {table_name}", conn)['n'].iloc[0]
print(f"\nTotal rows: {total:,}")

conn.close()

=== Track Metadata DB ===
Tables: ['songs']

Schema of 'songs':
              name type
          track_id TEXT
             title TEXT
           song_id TEXT
           release TEXT
         artist_id TEXT
       artist_mbid TEXT
       artist_name TEXT
          duration REAL
artist_familiarity REAL
 artist_hotttnesss REAL
              year  INT
  track_7digitalid  INT
          shs_perf  INT
          shs_work  INT

Sample rows:
             track_id              title             song_id  \
0  TRMMMYQ128F932D901       Silent Night  SOQMMHC12AB0180CB8   
1  TRMMMKD128F425225D        Tanssi vaan  SOVFVAK12A8C1350D9   
2  TRMMMRX128F93187D9  No One Could Ever  SOGTUKN12AB017F4F1   
3  TRMMMCH128F425532C      Si Vos Querés  SOBNYVR12A8C13558C   
4  TRMMMWA128F426B589   Tangle Of Aspens  SOHSBXH12A8C13B0DF   

                                release           artist_id  \
0                 Monster Ballads X-Mas  ARYZTJS1187B98C555   
1                           Karkuteillä  ARMVN3U118

In [ ]:
# ── Explore: Last.fm Zip files ────────────────────────────────────────────
# The Last.fm MSD dataset maps song IDs to Last.fm tags.
# Format varies: could be a SQLite DB (.db), TSV, or HDF5 inside the zip.

def inspect_zip(path, label):
    print(f"\n=== {label} ===")
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        print(f"  Files inside ({len(names)}):")
        for n in names[:20]:
            info = zf.getinfo(n)
            print(f"    {n}  ({info.file_size/1e6:.1f} MB)")
        # Peek at first text-like file
        for n in names:
            if n.endswith(('.txt', '.tsv', '.csv')):
                print(f"\n  First 3 lines of '{n}':")
                with zf.open(n) as f:
                    for i, line in enumerate(f):
                        print("   ", line.decode('utf-8', errors='replace').strip())
                        if i >= 2: break
                break

inspect_zip(LASTFM_TRAIN_ZIP, "lastfm_train.zip")
inspect_zip(LASTFM_TEST_ZIP,  "lastfm_test.zip")

## 2. Step 1 — Build `user_data_final.csv`

Source: `train_triplets.txt.zip` (Echo Nest taste profile)

Target columns: `user`, `song_id`, `play_count`

Key decisions from notebook analysis:
- Keep one row per `(user, song_id)` pair
- `play_count` must be a positive integer
- `user` and `song_id` must be non-null strings
- Aggregate duplicate `(user, song_id)` by summing play counts

In [ ]:
# ── Read train_triplets in chunks to stay within memory limits ─────────────
# Memory strategy:
#   1. Read CHUNK_SIZE rows at a time
#   2. Downcast dtypes immediately
#   3. Accumulate chunk summaries, then reduce once at the end

print("Reading Echo Nest triplets (chunked)…")

chunk_list = []

with zipfile.ZipFile(TRIPLETS_ZIP) as zf:
    inner_name = zf.namelist()[0]
    with zf.open(inner_name) as raw_file:
        reader = pd.read_csv(
            raw_file,
            sep='\t',
            header=None,
            names=['user', 'song_id', 'play_count'],
            dtype={'user': 'str', 'song_id': 'str', 'play_count': 'int32'},
            chunksize=CHUNK_SIZE
        )
        for chunk in tqdm(reader, desc="Triplets chunks"):
            # ── Data cleaning per chunk ────────────────────────────────
            chunk.dropna(subset=['user','song_id','play_count'], inplace=True)
            chunk = chunk[chunk['play_count'] >= MIN_PLAY_COUNT]

            # Aggregate duplicates within the chunk
            chunk = chunk.groupby(['user','song_id'], as_index=False)['play_count'].sum()
            chunk['play_count'] = chunk['play_count'].astype('int32')

            chunk_list.append(chunk)

print(f"  Chunks collected: {len(chunk_list)}")

In [ ]:
# ── Global reduction: sum play counts across chunks for the same (user, song) ─
print("Reducing all chunks…")
user_data = pd.concat(chunk_list, ignore_index=True)
del chunk_list
gc.collect()

user_data = user_data.groupby(['user','song_id'], as_index=False)['play_count'].sum()
user_data['play_count'] = user_data['play_count'].astype('int32')

df_mem(user_data, 'user_data')
print(f"  Unique users       : {user_data['user'].nunique():,}")
print(f"  Unique songs       : {user_data['song_id'].nunique():,}")
print(f"  Total interactions : {len(user_data):,}")
user_data.head()

In [ ]:
# ── Filter: remove users with too few songs (sparse users hurt quality) ────
print("Filtering sparse users…")
songs_per_user = user_data.groupby('user')['song_id'].transform('count')
user_data = user_data[songs_per_user >= MIN_SONGS_PER_USER].reset_index(drop=True)

print(f"  After filter — users: {user_data['user'].nunique():,}  "
      f"interactions: {len(user_data):,}")

# ── Validation ─────────────────────────────────────────────────────────────
assert user_data['user'].notna().all(),     "NULL user ids found!"
assert user_data['song_id'].notna().all(),  "NULL song ids found!"
assert (user_data['play_count'] > 0).all(), "Non-positive play counts found!"
assert user_data.duplicated(['user','song_id']).sum() == 0, \
    "Duplicate (user, song) pairs found!"

print("  Validation PASSED.")
user_data.describe()

In [ ]:
# ── Save user_data_final.csv ───────────────────────────────────────────────
user_data.to_csv(OUT_USER_DATA, index=False)
print(f"Saved: {OUT_USER_DATA}")
print(user_data.head())

## 3. Step 2 — Build `songs_final.csv`

Target columns: `song_id`, `artist_name`, `release`, `title`, `tags`, `arousal`, `valence`, `lyrics_text`

Sources:
- `track_metadata.db` → `song_id`, `artist_name`, `release`, `title`
- `lastfm_train.zip` + `lastfm_test.zip` → `tags` (stringified Python list, e.g. `"['rock', 'pop']"`)  
- `arousal`, `valence` → filled with `0.5` by default (merge from external MER data if available)
- `lyrics_text` → filled with `""` by default (merge from musiXmatch if available)

**Tag format requirement** (from notebook):
```python
tags = eval(song_data['tags'].iloc[0])   # => list of strings
```
→ Must be stored as a **stringified Python list**: `"['tag1', 'tag2']"` (with single quotes inside)

In [ ]:
# ── Load track metadata from SQLite in batches ────────────────────────────
# We only keep songs that appear in user_data (listened songs)

print("Loading track metadata from SQLite…")
listened_song_ids = set(user_data['song_id'].unique())
print(f"  Distinct listened song ids: {len(listened_song_ids):,}")

conn = sqlite3.connect(TRACK_META_DB)

# Discover the songs table and its columns
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
SONGS_TABLE = tables['name'].iloc[0]   # typically 'songs'
schema = pd.read_sql(f"PRAGMA table_info({SONGS_TABLE})", conn)
available_cols = schema['name'].tolist()
print(f"  Table: '{SONGS_TABLE}'  columns: {available_cols}")

# Map standard target column names to actual DB column names (adjust if needed)
COL_MAP = {
    'song_id'    : 'song_id',
    'title'      : 'title',
    'release'    : 'release',
    'artist_name': 'artist_name',
}

# Validate mapping against actual schema
for target, actual in COL_MAP.items():
    if actual not in available_cols:
        raise ValueError(f"Column '{actual}' not found in DB. Available: {available_cols}")

select_cols = ', '.join(COL_MAP.values())
print(f"  Selecting: {select_cols}")

# Batch-read to avoid loading 1M+ rows at once
BATCH = 200_000
offset = 0
meta_chunks = []

while True:
    sql = f"""
        SELECT {select_cols}
        FROM   {SONGS_TABLE}
        LIMIT  {BATCH}
        OFFSET {offset}
    """
    batch = pd.read_sql(sql, conn)
    if batch.empty:
        break
    # Keep only songs present in user_data
    batch = batch[batch['song_id'].isin(listened_song_ids)]
    if not batch.empty:
        meta_chunks.append(batch)
    offset += BATCH
    print(f"  … processed {offset:,} rows, kept {sum(len(c) for c in meta_chunks):,}")

conn.close()

song_data = pd.concat(meta_chunks, ignore_index=True)
del meta_chunks
gc.collect()

# Rename to standard names
rename = {v: k for k, v in COL_MAP.items() if v != k}
song_data.rename(columns=rename, inplace=True)

# Deduplicate on song_id
song_data.drop_duplicates(subset=['song_id'], keep='first', inplace=True)
song_data.reset_index(drop=True, inplace=True)

df_mem(song_data, 'song_data')
print(f"  Songs matched to user_data: {len(song_data):,}")
song_data.head()

In [ ]:
# ── Clean track metadata ───────────────────────────────────────────────────

# Fill missing string fields
for col in ['artist_name', 'release', 'title']:
    song_data[col] = song_data[col].fillna('').astype(str).str.strip()

# Remove rows where song_id is null (shouldn't happen, but guard anyway)
song_data = song_data[song_data['song_id'].notna() & (song_data['song_id'] != '')]

print(f"  Songs after cleaning: {len(song_data):,}")
print("  Null counts:")
print(song_data[['song_id','artist_name','release','title']].isna().sum())

### 3a. Parse Last.fm Data for Tags

The Last.fm MSD subset can come in several formats. Run the auto-detect cell below — it will pick the correct parser.

**Known formats:**
1. **SQLite DB** (`.db` file inside zip): tables `tids` (track IDs) and `tags` joined by `tid_tag`
2. **TSV/TXT** with columns `user \t timestamp \t artist \t track` (listening history) — tags not directly available, tags are approximated from artist names in this case
3. **TSV** with columns `track_id \t tag \t count` (tag counts file)

In [ ]:
# ── Auto-detect Last.fm zip format and extract song→tags mapping ───────────

def detect_lastfm_format(zip_path):
    """Returns one of: 'sqlite', 'tag_tsv', 'listen_tsv', 'unknown'"""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        for n in names:
            if n.endswith('.db'):           return 'sqlite', n
            if 'tag' in n.lower() and n.endswith(('.txt','.tsv','.csv')):
                return 'tag_tsv', n
        for n in names:
            if n.endswith(('.txt','.tsv','.csv')):
                return 'listen_tsv', n
    return 'unknown', None

fmt_train, inner_train = detect_lastfm_format(LASTFM_TRAIN_ZIP)
fmt_test,  inner_test  = detect_lastfm_format(LASTFM_TEST_ZIP)
print(f"Train zip format: {fmt_train}  ({inner_train})")
print(f"Test  zip format: {fmt_test}  ({inner_test})")

In [ ]:
# ── Parser A: SQLite format ────────────────────────────────────────────────
# Expected tables: 'tids' (tid, tid_str), 'tags' (tag, tag_id), 'tid_tag' (tid, tag_id, val)
# This is the standard Last.fm MSD database format.

import tempfile, shutil

def parse_lastfm_sqlite(zip_path, inner_name, listened_ids):
    """
    Extract a song_id → list-of-tags mapping from the Last.fm SQLite DB.
    Returns: dict  {song_id_str: [tag1, tag2, ...]}
    """
    # Extract DB to a temp file (SQLite can't be opened directly from a zip stream)
    tmp_dir = tempfile.mkdtemp()
    tmp_db  = os.path.join(tmp_dir, 'lastfm.db')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            with zf.open(inner_name) as src, open(tmp_db, 'wb') as dst:
                shutil.copyfileobj(src, dst)

        conn = sqlite3.connect(tmp_db)

        # List available tables
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
        print(f"  Tables in {inner_name}: {tables['name'].tolist()}")

        # Build song→tags via JOIN (chunked to save memory)
        sql = """
            SELECT t.tid_str AS song_id,
                   tg.tag   AS tag
            FROM   tids    t
            JOIN   tid_tag tt ON t.tid    = tt.tid
            JOIN   tags    tg ON tt.tag_id = tg.tag_id
            WHERE  tt.val > 0
        """
        song_tags = {}
        BATCH = 500_000
        offset = 0
        while True:
            batch = pd.read_sql(sql + f" LIMIT {BATCH} OFFSET {offset}", conn)
            if batch.empty:
                break
            batch = batch[batch['song_id'].isin(listened_ids)]
            for _, row in batch.iterrows():
                song_tags.setdefault(row['song_id'], set()).add(
                    row['tag'].lower().strip()
                )
            offset += BATCH
            print(f"    … processed {offset:,} tag rows")
        conn.close()
    finally:
        shutil.rmtree(tmp_dir)

    return {k: sorted(v) for k, v in song_tags.items()}


# ── Parser B: TSV tag file format ─────────────────────────────────────────
# Columns: track_id <TAB> tag <TAB> count  (or similar)

def parse_lastfm_tag_tsv(zip_path, inner_name, listened_ids):
    song_tags = {}
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(inner_name) as f:
            reader = pd.read_csv(
                f, sep='\t', header=None,
                names=['song_id','tag','count'],
                dtype=str,
                chunksize=CHUNK_SIZE
            )
            for chunk in tqdm(reader, desc="Tag TSV"):
                chunk.dropna(subset=['song_id','tag'], inplace=True)
                chunk = chunk[chunk['song_id'].isin(listened_ids)]
                for _, row in chunk.iterrows():
                    song_tags.setdefault(row['song_id'], set()).add(
                        str(row['tag']).lower().strip()
                    )
    return {k: sorted(v) for k, v in song_tags.items()}


# ── Parser C: Listening history TSV (no direct tags) ──────────────────────
# Columns: user <TAB> timestamp <TAB> artist <TAB> track
# We cannot derive song_id→tags directly from this format.
# Instead we return an empty dict; tags will be left as empty lists.

def parse_lastfm_listen_tsv(zip_path, inner_name, listened_ids):
    print("  WARNING: Listening history format detected — tags not extractable from this file.")
    print("  Songs will receive empty tag lists. Provide a Last.fm tag database for better results.")
    return {}


print("Parsers defined.")

In [ ]:
# ── Run the appropriate parser for each zip file ───────────────────────────

PARSERS = {
    'sqlite'    : parse_lastfm_sqlite,
    'tag_tsv'   : parse_lastfm_tag_tsv,
    'listen_tsv': parse_lastfm_listen_tsv,
}

song_tags = {}   # accumulate from both train and test zips

for zip_path, fmt, inner in [
    (LASTFM_TRAIN_ZIP, fmt_train, inner_train),
    (LASTFM_TEST_ZIP,  fmt_test,  inner_test),
]:
    if fmt == 'unknown':
        print(f"WARNING: Cannot parse {zip_path} — format unknown, skipping.")
        continue
    print(f"\nParsing {zip_path} (format={fmt})…")
    partial = PARSERS[fmt](zip_path, inner, listened_song_ids)
    print(f"  → Got tags for {len(partial):,} songs")
    # Merge: union of tags from train and test
    for sid, tags in partial.items():
        existing = song_tags.get(sid, [])
        song_tags[sid] = sorted(set(existing) | set(tags))

print(f"\nTotal songs with at least one tag: {len(song_tags):,}")

In [ ]:
# ── Merge tags into song_data ──────────────────────────────────────────────
#
# Target format (from notebook analysis):
#   song_data['tags'] stores a **stringified Python list** so that
#   downstream code can call:  eval(song_data['tags'].iloc[0])  → ['tag1','tag2']

print("Merging tags into song_data…")

song_data['tags'] = song_data['song_id'].map(song_tags)

# Songs with no tags get an empty list
no_tag_count = song_data['tags'].isna().sum()
song_data['tags'] = song_data['tags'].apply(
    lambda x: x if isinstance(x, list) else []
)

print(f"  Songs with no tags: {no_tag_count:,}")
print(f"  Songs with tags   : {(song_data['tags'].apply(len) > 0).sum():,}")

# Convert list to stringified Python list (REQUIRED by downstream notebooks)
# e.g.  ['rock', 'pop']  →  stored as string "['rock', 'pop']"
song_data['tags'] = song_data['tags'].apply(lambda x: str(x))

# Verify round-trip eval works
sample_tag_str = song_data['tags'].iloc[0]
try:
    evaled = eval(sample_tag_str)
    assert isinstance(evaled, list), "eval() did not return a list!"
    print(f"  Tag format verification PASSED. Sample: {sample_tag_str}")
except Exception as e:
    print(f"  Tag format verification FAILED: {e}")

song_data.head()

### 3b. `arousal` & `valence` Columns

These are song-level emotion features used for mood-based recommendation.
The HEMR notebooks compute user mood as a **play-count-weighted average** over listened songs.

**If you have an external mood dataset** (e.g. MER, Emotify, AllMusic mood tags), run the optional cell below.
Otherwise, the default cell fills both columns with `0.5` (neutral midpoint of a [0,1] scale),
which ensures the pipeline runs end-to-end while producing uniform (non-informative) mood features.

In [ ]:
# ── Default: fill arousal & valence with neutral value ────────────────────
# Change this to False if you are supplying an external mood file below.
USE_DEFAULT_MOOD = True

if USE_DEFAULT_MOOD:
    print("Using default arousal=0.5, valence=0.5 for all songs.")
    song_data['arousal'] = np.float32(0.5)
    song_data['valence'] = np.float32(0.5)
    print("  ↳ To improve recommendation quality, provide an actual mood dataset.")

In [ ]:
# ── OPTIONAL: Merge mood data from an external CSV ─────────────────────────
# Expected columns: song_id, arousal (float [0,1]), valence (float [0,1])
#
# MOOD_CSV = f"{RAW_DIR}/mood_data.csv"   # ← set your actual path
#
# if not USE_DEFAULT_MOOD:
#     mood_df = pd.read_csv(MOOD_CSV, dtype={'song_id':'str','arousal':'float32','valence':'float32'})
#     mood_df = mood_df[['song_id','arousal','valence']].drop_duplicates('song_id')
#
#     # Merge and fall back to 0.5 for unmatched songs
#     song_data = song_data.merge(mood_df, on='song_id', how='left')
#     song_data['arousal'] = song_data['arousal'].fillna(0.5).astype('float32')
#     song_data['valence'] = song_data['valence'].fillna(0.5).astype('float32')
#
#     # Validate range
#     assert song_data['arousal'].between(0, 1).all(), "arousal out of [0,1] range!"
#     assert song_data['valence'].between(0, 1).all(), "valence out of [0,1] range!"
#     print(f"Mood data merged. {mood_df['song_id'].isin(song_data['song_id']).sum():,} songs matched.")

print("(Mood merge cell — uncomment and configure if you have mood data)")

### 3c. `lyrics_text` Column

Used for BERT content embeddings in the downstream notebook.

**If you have the musiXmatch dataset or another lyrics source**, run the optional cell.
Otherwise the column is filled with an empty string — the BERT embedding step will still run
(producing a zero/uniform embedding) and the pipeline remains functional.

In [ ]:
# ── Default: empty lyrics ─────────────────────────────────────────────────
USE_DEFAULT_LYRICS = True

if USE_DEFAULT_LYRICS:
    print("Using empty lyrics_text for all songs.")
    song_data['lyrics_text'] = ''
    print("  ↳ To enable content embeddings, provide a lyrics dataset (e.g. musiXmatch).")

In [ ]:
# ── OPTIONAL: Merge lyrics from an external CSV ────────────────────────────
# Expected columns: song_id (str), lyrics_text (str)
#
# LYRICS_CSV = f"{RAW_DIR}/lyrics.csv"   # ← set your actual path
#
# if not USE_DEFAULT_LYRICS:
#     # Chunked merge to handle large lyrics files
#     lyrics_chunks = []
#     for chunk in pd.read_csv(LYRICS_CSV, dtype={'song_id':'str'}, chunksize=CHUNK_SIZE):
#         chunk = chunk[['song_id','lyrics_text']]
#         chunk = chunk[chunk['song_id'].isin(listened_song_ids)]
#         chunk['lyrics_text'] = chunk['lyrics_text'].fillna('').str.replace(r'\n', ' ', regex=True)
#         lyrics_chunks.append(chunk)
#
#     lyrics_df = pd.concat(lyrics_chunks, ignore_index=True).drop_duplicates('song_id')
#     del lyrics_chunks; gc.collect()
#
#     song_data = song_data.merge(lyrics_df, on='song_id', how='left')
#     song_data['lyrics_text'] = song_data['lyrics_text'].fillna('')
#     print(f"Lyrics merged. {(song_data['lyrics_text'] != '').sum():,} songs have lyrics.")

print("(Lyrics merge cell — uncomment and configure if you have a lyrics file)")

In [ ]:
# ── Finalise songs_final.csv ───────────────────────────────────────────────

# Enforce exact column order expected by the HEMR notebooks
TARGET_COLS = ['song_id', 'artist_name', 'release', 'title',
               'tags', 'arousal', 'valence', 'lyrics_text']

# Add any missing columns (shouldn't happen if earlier steps ran correctly)
for col in TARGET_COLS:
    if col not in song_data.columns:
        print(f"  WARNING: column '{col}' missing — adding default.")
        if col in ('arousal','valence'):
            song_data[col] = np.float32(0.5)
        elif col == 'lyrics_text':
            song_data[col] = ''
        elif col == 'tags':
            song_data[col] = '[]'
        else:
            song_data[col] = ''

song_data = song_data[TARGET_COLS].copy()

# ── Final validation ───────────────────────────────────────────────────────
assert song_data['song_id'].notna().all(),     "NULL song_ids in songs_final!"
assert song_data.duplicated('song_id').sum() == 0, "Duplicate song_ids!"
assert song_data['tags'].apply(lambda x: isinstance(eval(x), list)).all(), \
    "tags column is not a valid stringified list!"
assert song_data['arousal'].between(0, 1).all(), "arousal out of range!"
assert song_data['valence'].between(0, 1).all(), "valence out of range!"

print("Validation PASSED.")
df_mem(song_data, 'song_data')
print(f"  Total songs: {len(song_data):,}")
song_data.head()

In [ ]:
# ── Save songs_final.csv ───────────────────────────────────────────────────
song_data.to_csv(OUT_SONGS, index=False)
print(f"Saved: {OUT_SONGS}")
print(song_data.dtypes)
song_data.head()

## 4. Step 3 — Cross-filter & Align

The downstream notebooks assume:
- Every `song_id` in `user_data` has a corresponding row in `song_data` (and vice-versa)
- i.e. the two datasets share exactly the same song universe

In [ ]:
# ── Cross-filter to ensure referential integrity ───────────────────────────

print("Before cross-filter:")
print(f"  user_data songs : {user_data['song_id'].nunique():,}")
print(f"  song_data songs : {len(song_data):,}")

# Keep only songs that exist in BOTH datasets
common_songs = set(user_data['song_id']) & set(song_data['song_id'])
user_data = user_data[user_data['song_id'].isin(common_songs)].copy()
song_data = song_data[song_data['song_id'].isin(common_songs)].copy()

# Re-filter users (some may now have too few songs after the song filter)
songs_per_user = user_data.groupby('user')['song_id'].transform('count')
user_data = user_data[songs_per_user >= MIN_SONGS_PER_USER].reset_index(drop=True)

print("After cross-filter:")
print(f"  Common songs     : {len(common_songs):,}")
print(f"  user_data rows   : {len(user_data):,}")
print(f"  song_data rows   : {len(song_data):,}")
print(f"  Unique users     : {user_data['user'].nunique():,}")

In [ ]:
# ── Re-save user_data_final after cross-filter ─────────────────────────────
user_data.to_csv(OUT_USER_DATA, index=False)
print(f"Re-saved: {OUT_USER_DATA}")

# Re-save song_data (now reduced to exactly the listened song universe)
song_data.reset_index(drop=True, inplace=True)
song_data.to_csv(OUT_SONGS, index=False)
print(f"Re-saved: {OUT_SONGS}")

## 5. Step 4 — Train/Test Split → `user_data_partitioned_val.csv`

Matching the logic from the HEMR notebooks exactly:
- Each user's interactions are shuffled randomly
- Users with **< 5 interactions** → all assigned to **train**
- Others → 80% train, 20% test (per user)
- A `triple_id` column (global row index) is added
- A `set` column stores `"train"` or `"test"`

In [ ]:
import gc
import numpy as np

# ── Add triple_id (same as before) ────────────────────────────────────────
# Avoid .copy() — work directly to save one full DataFrame copy in RAM.
user_data_part = user_data  # no copy; we only add columns, never mutate existing ones
user_data_part = user_data_part.reset_index(drop=True)
user_data_part['triple_id'] = user_data_part.index.astype('int32')

# ── Memory problem in the original approach ────────────────────────────────
# groupby.apply(assign_split) calls a Python function per group, building
# set_dict with millions of entries via a row-level for-loop. This:
#   1. Holds the entire user_data_part + all group slices in RAM simultaneously
#   2. Builds a Python dict with O(n_interactions) entries (one per row)
#   3. A final .map(set_dict) then allocates yet another full Series
#
# ── Fix: fully vectorised numpy split — zero Python loops ─────────────────
# Strategy:
#   1. Sort by user so each user's rows are contiguous (enables vectorised ops)
#   2. Compute a per-row within-user rank using cumcount()
#   3. Compute each user's interaction count via transform('count')
#   4. Derive train/test label arithmetically — no dict, no apply, no loop
#   5. Downcast to category to halve the 'set' column's memory footprint

# Step 1: sort so each user's rows sit together; use mergesort (stable)
# so the shuffle within each user group is reproducible.
# We shuffle by assigning a random key per row, then sorting on (user, rand_key).
rng = np.random.default_rng(RANDOM_SEED)
user_data_part['_rand'] = rng.random(len(user_data_part)).astype('float32')

user_data_part.sort_values(['user', '_rand'], inplace=True)
user_data_part.reset_index(drop=True, inplace=True)

# Step 2: within-user row rank (0-based) and group size — both O(n), vectorised
user_data_part['_rank']  = user_data_part.groupby('user').cumcount()          # int64
user_data_part['_total'] = user_data_part.groupby('user')['user'].transform('count')  # int64

# Downcast immediately to free memory
user_data_part['_rank']  = user_data_part['_rank'].astype('int32')
user_data_part['_total'] = user_data_part['_total'].astype('int32')

# Step 3: compute the cut-point for each user (number of train rows)
# Mirrors original logic:
#   if total < MIN_TRAIN_ITEMS  → all train  (cut = total)
#   else                        → cut = round(total * TRAIN_RATIO)
cut = np.where(
    user_data_part['_total'] < MIN_TRAIN_ITEMS,
    user_data_part['_total'],                                    # all train
    (user_data_part['_total'] * TRAIN_RATIO).round().astype('int32')
)

# Step 4: assign label — single vectorised comparison, no dict needed
# Convert to pandas Series before applying 'category' dtype
user_data_part['set'] = pd.Series(
    np.where(user_data_part['_rank'] < cut, 'train', 'test')
).astype('category')   # category saves ~5x vs plain object strings

# Step 5: drop helper columns and free memory immediately
user_data_part.drop(columns=['_rand', '_rank', '_total'], inplace=True)
del cut
gc.collect()

# ── Sanity checks (same as before) ────────────────────────────────────────
assert user_data_part['set'].notna().all(), "Some rows have no train/test label!"

train_n = (user_data_part['set'] == 'train').sum()
test_n  = (user_data_part['set'] == 'test').sum()
print(f"  Train rows : {train_n:,}  ({100*train_n/len(user_data_part):.1f}%)")
print(f"  Test rows  : {test_n:,}  ({100*test_n/len(user_data_part):.1f}%)")
print(f"  Memory     : {user_data_part.memory_usage(deep=True).sum()/1e6:.1f} MB")

In [ ]:
# ── Save partitioned dataset ───────────────────────────────────────────────
# Column order matching the HEMR notebooks' expectations:
# (row index saved as the first unnamed column — notebooks use .iloc[:,0] as index)
user_data_part.to_csv(OUT_USER_PART, index=True)   # index=True matches original format
print(f"Saved: {OUT_USER_PART}")
print(user_data_part.head())
print(user_data_part.dtypes)

## 6. Final Verification & Summary

In [ ]:
# ── Reload CSVs and verify format exactly matches notebook expectations ────
print("Reloading saved CSVs for verification…\n")

v_songs     = pd.read_csv(OUT_SONGS)
v_user      = pd.read_csv(OUT_USER_DATA)
v_user_part = pd.read_csv(OUT_USER_PART)

print("=== songs_final.csv ===")
print(f"  Shape  : {v_songs.shape}")
print(f"  Columns: {v_songs.columns.tolist()}")
print(f"  Dtypes :\n{v_songs.dtypes}")
print(f"  Nulls  :\n{v_songs.isna().sum()}")
print(v_songs.head(3))

print("\n=== user_data_final.csv ===")
print(f"  Shape  : {v_user.shape}")
print(f"  Columns: {v_user.columns.tolist()}")
print(f"  Dtypes :\n{v_user.dtypes}")
print(f"  Nulls  :\n{v_user.isna().sum()}")
print(v_user.head(3))

print("\n=== user_data_partitioned_val.csv ===")
print(f"  Shape  : {v_user_part.shape}")
print(f"  Columns: {v_user_part.columns.tolist()}")
print(f"  set dist:\n{v_user_part['set'].value_counts()}")
print(v_user_part.head(3))

In [ ]:
# ── Hard assertions: guarantee downstream notebooks can run ────────────────

# songs_final.csv
required_song_cols = ['song_id','artist_name','release','title','tags','arousal','valence','lyrics_text']
assert all(c in v_songs.columns for c in required_song_cols), \
    f"Missing columns in songs_final: {set(required_song_cols)-set(v_songs.columns)}"
assert v_songs['song_id'].notna().all(),  "song_id has nulls"
assert v_songs.duplicated('song_id').sum() == 0, "Duplicate song_ids"
# Verify tags are eval-able lists
sample_eval = eval(v_songs['tags'].iloc[0])
assert isinstance(sample_eval, list), "tags column must evaluate to a Python list"

# user_data_final.csv
required_user_cols = ['user','song_id','play_count']
assert all(c in v_user.columns for c in required_user_cols), \
    f"Missing columns in user_data_final: {set(required_user_cols)-set(v_user.columns)}"
assert v_user['user'].notna().all(),    "user has nulls"
assert v_user['song_id'].notna().all(), "song_id has nulls"
assert (v_user['play_count'] > 0).all(), "play_count must be > 0"

# Referential integrity
unmatched = set(v_user['song_id']) - set(v_songs['song_id'])
assert len(unmatched) == 0, f"{len(unmatched)} song_ids in user_data not in songs_final"

# user_data_partitioned_val.csv
required_part_cols = ['user','song_id','play_count','triple_id','set']
assert all(c in v_user_part.columns for c in required_part_cols), \
    f"Missing columns in partitioned: {set(required_part_cols)-set(v_user_part.columns)}"
assert set(v_user_part['set'].dropna().unique()) <= {'train','test'}, \
    "'set' column has values other than 'train'/'test'"

print("ALL ASSERTIONS PASSED ✅")
print("\nThe preprocessed datasets are ready for the HEMR notebooks.")

In [ ]:
# ── Summary Statistics ─────────────────────────────────────────────────────

print("╔══════════════════════════════════════════════════════════╗")
print("║              Preprocessing Summary                      ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Unique songs          : {len(v_songs):>10,}                   ║")
print(f"║  Unique users          : {v_user['user'].nunique():>10,}                   ║")
print(f"║  Total interactions    : {len(v_user):>10,}                   ║")
print(f"║  Songs with tags       : {(v_songs['tags'] != '[]').sum():>10,}                   ║")
print(f"║  Train interactions    : {(v_user_part['set']=='train').sum():>10,}                   ║")
print(f"║  Test  interactions    : {(v_user_part['set']=='test').sum():>10,}                   ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Output: {OUTPUT_DIR:<47}║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  songs_final.csv             ← song metadata + tags    ║")
print(f"║  user_data_final.csv         ← user/song interactions  ║")
print(f"║  user_data_partitioned_val.csv ← with train/test split ║")
print("╚══════════════════════════════════════════════════════════╝")

print("\nPlay count distribution:")
print(v_user['play_count'].describe())
print("\nSongs per user distribution:")
print(v_user.groupby('user')['song_id'].count().describe())

## 7. Next Steps

1. **In your HEMR notebooks**, update the CSV paths:
   ```python
   song_data = pd.read_csv("drive/MyDrive/HEMR_preprocessed/songs_final.csv")
   user_data = pd.read_csv("drive/MyDrive/HEMR_preprocessed/user_data_final.csv")
   # or for the partitioned version:
   user_data_total = pd.read_csv("drive/MyDrive/HEMR_preprocessed/user_data_partitioned_val.csv")
   ```

2. **Improve `arousal` / `valence`** — merge from an external MER dataset for mood-aware recommendations.

3. **Improve `lyrics_text`** — merge from the musiXmatch dataset for BERT content embeddings.

4. **Improve `tags`** — if `lastfm_train.zip` contained a listening history rather than a tag DB,
   consider downloading the Last.fm MSD tag database separately.

### Column Reference

| Column | Used for | Type |
|---|---|---|
| `song_data['tags']` | Tag hyperedges in the hypergraph | `str` (eval → `list`) |
| `song_data['arousal']` | User mood computation | `float` [0,1] |
| `song_data['valence']` | User mood computation | `float` [0,1] |
| `song_data['lyrics_text']` | BERT content embeddings | `str` |
| `user_data['play_count']` | Weighted mood avg, user hyperedges | `int` |
| `user_data_total['set']` | Train/test evaluation split | `'train'`/`'test'` |